| Device | PC connection |
|---|---|
| RIGOL DSG836/DSG836A generator | USB via PyVISA |
| Tektronix TDS2000C scope | USB via PyVISA |
| Trigger box | **COM8**, 9600 baud |
| Newport ESP stage controller | **COM3**

In [ ]:
# Run once if packages are missing:
# %pip install numpy ipywidgets pyvisa pyserial

**Connection Check**

In [1]:
import serial

with serial.Serial("COM8", 9600, timeout=1):
    print("COM8 opened successfully")

COM8 opened successfully


In [1]:
import math, re
import pyvisa, serial
from newportESP import ESP

if 'start' in globals() and start.disabled:
    raise RuntimeError('Stop the scan first.')
CONNECTIONS_OK = False

RIGOL = 'USB0::0x1AB1::0x099C::DSG8M223900110::INSTR'
SCOPE = 'USB0::0x699::0x03A3::C021609::INSTR'
TRIGGER_PORT, STAGE_PORT = 'COM8', 'COM3'

rm = pyvisa.ResourceManager()
try:
    for name, address, model in [
        ('Generator', RIGOL, r'DSG836A?(?=[,\s]|$)'),
        ('Scope', SCOPE, r'TDS\s*20\d\dC(?=[,\s]|$)')
    ]:
        with rm.open_resource(address) as device:
            device.timeout = 1500
            device.write_termination = device.read_termination = '\n'
            identity = device.query('*IDN?').strip()
            if not re.search(model, identity, re.I):
                raise RuntimeError(f'{name}: unexpected identity: {identity}')
            print(f'OK {name}: {identity}')
finally:
    rm.close()

with serial.Serial(TRIGGER_PORT, 9600, timeout=1, write_timeout=1):
    print(f'OK Trigger: {TRIGGER_PORT} opens; pulse not tested')

motor = ESP(port=STAGE_PORT)
try:
    x, y = [float(motor.query('TP', axis=a)) for a in (2, 1)]
    if not all(map(math.isfinite, (x, y))):
        raise RuntimeError('Invalid stage position')
    print(f'OK Stage: {STAGE_PORT}; X={x:g}, Y={y:g} mm')
finally:
    for attr in ('ser', 'serial', '_ser', '_serial'):
        port = getattr(motor, attr, None)
        if callable(getattr(port, 'close', None)):
            port.close()
            break

CONNECTIONS_OK = True
print('Connection checks passed.')

OK Generator: Rigol Technologies,DSG836,DSG8M223900110,00.01.07
OK Scope: TEKTRONIX,TDS 2012C,C021609,CF:91.1CT FV:v24.18
OK Trigger: COM8 opens; pulse not tested
OK Stage: COM3; X=0.17813, Y=0.0509 mm
Connection checks passed.


**GUI**

In [3]:
import csv, json, re, time, threading
from contextlib import ExitStack, suppress
from datetime import datetime
from pathlib import Path
from html import escape
import numpy as np
import pyvisa, serial, ipywidgets as W
from IPython.display import display
from newportESP import ESP

if 'start' in globals() and start.disabled:
    raise RuntimeError('Stop the current scan before rerunning this cell.')

RIGOL = 'USB0::0x1AB1::0x099C::DSG8M223900110::INSTR'
SCOPE = 'USB0::0x0699::0x03A3::C021609::INSTR'
TRIGGER_PORT, STAGE_PORT = 'COM8', 'COM3'
spec = [('f0', 'Start frequency (MHz)', '2700'), ('f1', 'End frequency (MHz)', '2850'),
        ('nf', 'Frequency points', '51'), ('dwell', 'Dwell per point (s)', '0.05'),
        ('nx', 'Nx', '3'), ('ny', 'Ny', '3'), ('dx', 'X step (sample µm)', '20'),
        ('dy', 'Y step (sample µm)', '20'), ('cal', 'Calibration (µm/mm)', '196.48'),
        ('settle', 'Stage settling (s)', '0.2'), ('delay', 'Trigger-to-RF delay (s)', '0'),
        ('x0_mm', 'X0 origin (stage mm)', '0'), ('y0_mm', 'Y0 origin (stage mm)', '0')]
fields = {k: W.Text(value=v, description=label, style={'description_width': '175px'},
                   layout=W.Layout(width='340px')) for k, label, v in spec}
folder = W.Text(value='scan_data', description='Output folder')
start, stop = W.Button(description='Start', button_style='success'), W.Button(description='Stop', disabled=True)
progress = W.IntProgress(max=1)
status = W.HTML('Review the sweep settings and stage origin, then click Start.')
halt, loop = threading.Event(), get_ipython().kernel.io_loop

def show(message, count=None):
    def update():
        status.value = escape(str(message))
        if count is not None:
            progress.value = count
    loop.add_callback(update)

def check_stop():
    if halt.is_set():
        raise InterruptedError('Stopped; completed measurements are saved.')

def wait_until(test, seconds, message):
    deadline = time.monotonic() + seconds
    while True:
        check_stop()
        if test():
            return
        if time.monotonic() >= deadline:
            raise TimeoutError(message)
        halt.wait(0.02)

def number(reply):
    # Rigol may group frequency digits with spaces, for example 2.700 000 000 00GHz.
    normalized = re.sub(r'(?<=\d)[^\S\r\n]+(?=\d)', '', reply)
    match = re.fullmatch(r'\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[Ee][+-]?\d+)?)\s*(Hz|kHz|MHz|GHz|s|ms)?\s*', normalized, re.I)
    if not match:
        raise ValueError('Unexpected instrument reply: ' + reply)
    return float(match[1]) * {'': 1, 'hz': 1, 'khz': 1e3, 'mhz': 1e6, 'ghz': 1e9, 's': 1, 'ms': 1e-3}[(match[2] or '').lower()]

def close_motor(motor):
    for attr in ('ser', 'serial', '_ser', '_serial'):
        port = getattr(motor, attr, None)
        if callable(getattr(port, 'close', None)):
            port.close()
            return

def scan(p):
    phase = 'Instrument connections'
    last_generator_condition = None
    try:
        with ExitStack() as stack:
            rm = pyvisa.ResourceManager(); stack.callback(rm.close)
            gen, scope = [stack.enter_context(rm.open_resource(a)) for a in (RIGOL, SCOPE)]
            for device, name, model in (
                (gen, 'Rigol', r'DSG836A?(?=[,\s]|$)'),
                (scope, 'Scope', r'TDS\s*20\d\dC(?=[,\s]|$)'),
            ):
                phase = f'{name} connection'
                device.timeout = 2000
                device.write_termination = device.read_termination = '\n'
                identity = device.query('*IDN?').strip()
                if not re.search(model, identity, re.I):
                    raise RuntimeError(f'Unexpected instrument identity {identity!r}.')
            phase = 'Trigger box and stage connections'
            trigger = stack.enter_context(serial.Serial(TRIGGER_PORT, 9600, timeout=1, write_timeout=1))
            motor = ESP(port=STAGE_PORT); stack.callback(close_motor, motor)
            def cleanup():
                # Disable RF output while keeping the selected sweep settings.
                for device, command in ((gen, ':OUTP OFF'), (scope, 'ACQUIRE:STATE STOP')):
                    with suppress(Exception): device.write(command)
                for axis in (2, 1):
                    with suppress(Exception): motor.write('ST', axis=axis)
            stack.callback(cleanup)
            phase = 'Stage limits and units'
            origin = np.array([p['x0_mm'], p['y0_mm']])
            for a, length in ((2, (p['nx']-1)*p['dx']/p['cal']), (1, (p['ny']-1)*p['dy']/p['cal'])):
                if int(float(motor.query('MD', axis=a))) != 1 or int(float(motor.query('SN', axis=a))) != 2:
                    raise RuntimeError('Stage must be stationary with units set to millimetres.')
                lo, hi = [float(motor.query(c, axis=a)) for c in ('SL', 'SR')]
                base = origin[0 if a == 2 else 1]
                if not lo <= base <= base + length <= hi:
                    axis_name = 'X' if a == 2 else 'Y'
                    raise ValueError(
                        f'{axis_name} scan spans {base:g} to {base + length:g} mm. '
                        f'The stage limits are {lo:g} to {hi:g} mm.'
                    )
            phase = 'Rigol setup'
            show('Configuring Rigol frequency sweep')
            for command in [':OUTP OFF', ':SWE:STAT OFF', ':SWE:TYPE STEP', ':SWE:STEP:SPAC LIN', ':SWE:STEP:SHAP RAMP',
                            ':SWE:DIR FWD', f":SWE:STEP:STAR:FREQ {p['f0']}", f":SWE:STEP:STOP:FREQ {p['f1']}",
                            f":SWE:STEP:POIN {p['nf']}", f":SWE:STEP:DWEL {p['dwell']}", ':SWE:MODE CONT',
                            ':SWE:SWE:TRIG:TYPE EXT', ':SWE:POIN:TRIG:TYPE AUTO', ':INP:TRIG:SLOP POS', ':SWE:STAT FREQ']:
                gen.write(command)
            for command, requested in [('STAR:FREQ', p['f0']), ('STOP:FREQ', p['f1']), ('POIN', p['nf']), ('DWEL', p['dwell'])]:
                query = f':SWE:STEP:{command}?'
                reply = gen.query(query).strip()
                if not np.isclose(number(reply), requested, rtol=1e-7, atol=1e-9):
                    raise RuntimeError(f'{query} returned {reply!r} for requested value {requested:g}.')
            for command, expected in [('TYPE', 'STEP'), ('STEP:SPAC', 'LIN'), ('STEP:SHAP', 'RAMP'),
                                      ('DIR', 'FWD'), ('MODE', 'CONT'), ('SWE:TRIG:TYPE', 'EXT'),
                                      ('POIN:TRIG:TYPE', 'AUTO'), ('STAT', 'FREQ')]:
                reply = gen.query(f':SWE:{command}?').strip().upper()
                if not reply.startswith(expected):
                    raise RuntimeError(f'Rigol :SWE:{command}? returned {reply!r}; expected {expected}.')
            phase = 'Scope setup'
            show('Rigol confirmed Freq / Step / Cont. Configuring scope.')
            for command in ['HEADER OFF', 'ACQUIRE:STATE STOP', 'SELECT:CH1 ON', 'HORIZONTAL:VIEW MAIN',
                            'DISPLAY:FORMAT YT', 'ACQUIRE:MODE SAMPLE', 'ACQUIRE:STOPAFTER SEQUENCE',
                            'TRIGGER:MAIN:TYPE EDGE', 'TRIGGER:MAIN:MODE NORMAL', 'TRIGGER:MAIN:EDGE:SOURCE EXT',
                            'TRIGGER:MAIN:EDGE:SLOPE RISE', 'TRIGGER:MAIN:EDGE:COUPLING DC', 'TRIGGER:MAIN:LEVEL 1',
                            'DATA:SOURCE CH1', 'DATA:ENCDG ASCII', 'DATA:WIDTH 1', 'DATA:START 1', 'DATA:STOP 2500']:
                scope.write(command)
            for command, expected in [('ACQUIRE:MODE', 'SAMPLE'), ('ACQUIRE:STOPAFTER', 'SEQ'),
                                      ('TRIGGER:MAIN:MODE', 'NORMAL'), ('TRIGGER:MAIN:EDGE:SOURCE', 'EXT')]:
                reply = scope.query(f'{command}?').strip().upper()
                if not reply.startswith(expected):
                    raise RuntimeError(f'{command}? returned {reply!r}; expected {expected}.')
            duration = p['nf'] * p['dwell']
            scale = next((v for e in range(-6, 2) for v in (10.**e, 2.5*10.**e, 5*10.**e)
                          if v <= 50 and 9*v >= duration+p['delay']+2*p['dwell']), None)
            if scale is None:
                raise ValueError('Sweep exceeds the scope time window; shorten it.')
            scope.write(f'HORIZONTAL:MAIN:SCALE {scale}')
            scale = float(scope.query('HORIZONTAL:MAIN:SCALE?'))
            scope.write(f'HORIZONTAL:MAIN:POSITION {4*scale}')
            if 9*scale < duration+p['delay']+p['dwell'] or p['dwell'] < 4*(10*scale/2500):
                raise ValueError('Scope memory cannot resolve this sweep; reduce frequency points.')
            phase = 'Opening output files'
            out = Path(p['folder']).expanduser() / datetime.now().strftime('scan_%Y%m%d_%H%M%S_%f')
            out.mkdir(parents=True, exist_ok=False)
            (out/'settings.json').write_text(json.dumps(dict(p, origin_stage_mm=origin.tolist(),
                generator_sweep_mode='CONT', generator_sweep_trigger='EXT', generator_point_trigger='AUTO',
                scope_s_per_div=scale, voltage_method='Mean of middle half of each programmed dwell',
                frequency_timing='Nominal dwell + user RF delay; validate experimentally'), indent=2))
            datafile = stack.enter_context((out/'data.csv').open('w', newline=''))
            writer = csv.writer(datafile); writer.writerow(['x_position', 'y_position', 'frequency', 'voltage'])
            frequencies = np.linspace(p['f0'], p['f1'], p['nf'])
            sweeping = 8
            waiting_for_trigger = 32
            sweep_calculation = 256
            sweep_status_mask = sweeping | waiting_for_trigger | sweep_calculation

            def condition():
                nonlocal last_generator_condition
                last_generator_condition = int(gen.query(':STAT:OPER:COND?'))
                return last_generator_condition

            def generator_ready():
                return (condition() & sweep_status_mask) == waiting_for_trigger

            phase = 'Rigol RF output'
            check_stop()
            gen.write(':OUTP ON')
            count = 0
            for row in range(p['ny']):
                for col in (range(p['nx']) if row % 2 == 0 else reversed(range(p['nx']))):
                    check_stop()
                    spot = count + 1
                    phase = f'Stage movement at spot {spot}'
                    xy = np.array([col*p['dx'], row*p['dy']])
                    target = origin + xy/p['cal']
                    show(f'Moving to ({xy[0]:g}, {xy[1]:g}) µm')
                    for axis, position in zip((2, 1), target): motor.write(f'PA{position:.9f}', axis=axis)
                    wait_until(lambda: all(float(motor.query('MD', axis=a)) == 1 for a in (2, 1)), 30, 'Stage motion timeout.')
                    actual = [float(motor.query('TP', axis=a)) for a in (2, 1)]
                    if not np.allclose(actual, target, rtol=0, atol=0.001):
                        raise RuntimeError('Stage did not reach its target (tolerance 0.001 mm).')
                    if halt.wait(p['settle']): check_stop()
                    phase = f'Rigol arming at spot {spot}'
                    # Continuous mode waits for each external trigger automatically.
                    wait_until(generator_ready, 10, f'Spot {spot}: Rigol did not report waiting for the external trigger.')
                    phase = f'Scope arming at spot {spot}'
                    scope.write('ACQUIRE:STATE RUN')
                    wait_until(lambda: scope.query('TRIGGER:STATE?').strip().upper() == 'READY', 10*scale+10, 'Scope did not become READY.')
                    check_stop()
                    phase = f'Rigol readiness at spot {spot}'
                    if not generator_ready():
                        raise RuntimeError(f'Spot {spot}: Rigol left the external trigger wait state before the pulse.')
                    phase = f'Trigger box at spot {spot}'
                    trigger_sent = time.monotonic()
                    trigger.write(b'\x01'); trigger.flush()
                    phase = f'Rigol sweep at spot {spot}'
                    show(f'Acquiring position {count+1}/{p["nx"]*p["ny"]}')
                    wait_until(
                        lambda: condition() & sweeping,
                        3,
                        f'Spot {spot}: Rigol sweep start was not observed. Check TRIGGER IN or lengthen the sweep.',
                    )
                    wait_until(
                        generator_ready,
                        2 * duration + 10,
                        f'Spot {spot}: Rigol did not finish the sweep and return to trigger wait.',
                    )
                    end_seen = time.monotonic()-trigger_sent
                    phase = f'Scope acquisition at spot {spot}'
                    wait_until(
                        lambda: (
                            int(scope.query('BUSY?')) == 0
                            and int(scope.query('ACQUIRE:NUMACQ?')) == 1
                        ),
                        10 * scale + 10,
                        f'Spot {spot}: scope did not complete a fresh acquisition.',
                    )
                    phase = f'Scope waveform readout at spot {spot}'
                    raw = np.asarray(scope.query_ascii_values('CURVE?'))
                    x0, dt, pt, y0, ym, yo = [float(scope.query('WFMPRE:'+q+'?')) for q in ('XZE', 'XIN', 'PT_OFF', 'YZE', 'YMU', 'YOFF')]
                    t, voltage = x0+(np.arange(len(raw))-pt)*dt, (raw-yo)*ym+y0
                    phase = f'Waveform validation and saving at spot {spot}'
                    np.savez(out/f'position_{count+1:04d}.npz', time_s=t, voltage_V=voltage,
                             frequency_Hz=frequencies, frequency_start_time_s=p['delay']+np.arange(p['nf'])*p['dwell'],
                             x_um=xy[0], y_um=xy[1], actual_stage_mm=actual, pc_sweep_end_detected_s=end_seen)
                    if dt <= 0 or not np.isfinite(voltage).all() or t[0] > p['delay'] or t[-1] < duration+p['delay']:
                        raise RuntimeError('Scope record does not cover the sweep; raw data saved.')
                    means = []
                    for k in range(p['nf']):
                        window = voltage[(t >= p['delay']+(k+.25)*p['dwell']) & (t < p['delay']+(k+.75)*p['dwell'])]
                        if len(window) < 2:
                            raise RuntimeError('Too few samples per frequency; reduce points. Raw data saved.')
                        means.append(window.mean())
                    writer.writerows((xy[0], xy[1], f, v) for f, v in zip(frequencies, means)); datafile.flush()
                    count += 1
                    show(f'Saved {count}/{p["nx"]*p["ny"]}: {out}', count)
            show(f'Complete: {out}', count)
    except InterruptedError as error:
        show(str(error))
    except Exception as error:
        detail = ''
        if phase.startswith('Rigol') and last_generator_condition is not None:
            detail = f' Last Rigol operation status = {last_generator_condition}.'
        show(f'{phase} | {type(error).__name__}: {error}{detail}')
    finally:
        loop.add_callback(set_running, False)

def set_running(running):
    for widget in [*fields.values(), folder, start]: widget.disabled = running
    stop.disabled = not running

def launch(_):
    if start.disabled:
        return
    try:
        if not fields['x0_mm'].value.strip() or not fields['y0_mm'].value.strip():
            raise ValueError('Enter both X0 and Y0 in stage mm.')
        p = {k: float(w.value) for k, w in fields.items()}
        if not all(np.isfinite(list(p.values()))): raise ValueError('Use finite numbers.')
        for k in ('nf', 'nx', 'ny'):
            if p[k] != int(p[k]): raise ValueError(k+' must be an integer.')
            p[k] = int(p[k])
        p['f0'] *= 1e6; p['f1'] *= 1e6
        if not (9000 <= p['f0'] < p['f1'] and 2 <= p['nf'] <= 65535 and .02 <= p['dwell'] <= 100):
            raise ValueError('Use increasing frequencies ≥0.009 MHz, ≥2 points, and dwell 0.02–100 s.')
        if min(p['nx'], p['ny'], p['dx'], p['dy'], p['cal']) <= 0 or min(p['settle'], p['delay']) < 0:
            raise ValueError('Grid/calibration must be positive; settling/delay must be nonnegative.')
        p['folder'] = folder.value.strip()
        if not p['folder']: raise ValueError('Enter an output folder.')
        halt.clear(); set_running(True)
        progress.max, progress.value = p['nx']*p['ny'], 0
        threading.Thread(target=scan, args=(p,), daemon=True).start()
    except Exception as error:
        set_running(False)
        show(error)

start.on_click(launch)
stop.on_click(lambda _: (halt.set(), show('Stopping…')))
scan_fields = [fields[k] for k, _, _ in spec if k not in ('x0_mm', 'y0_mm')]
display(W.VBox([
    W.HBox([fields['x0_mm'], fields['y0_mm']]),
    W.HBox([W.VBox(scan_fields[:6]), W.VBox(scan_fields[6:])]),
    folder, W.HBox([start, stop]), progress, status,
]))
